<a href="https://colab.research.google.com/github/nig414/AML-experiments/blob/main/AML_Experiment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load the dataset
print("Please upload the 'iris_synthetic_data.csv' file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

In [ ]:
# 2. Preprocess the dataset
# Fill missing values
df = df.fillna(df.median(numeric_only=True))

# Encode target label
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

# Define Features and Target
X = df.drop('label', axis=1)
y = df['label']

# Scaling is important for Logistic Regression convergence
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

In [ ]:
# 3. Initialize the Model (Logistic Regression)
model = LogisticRegression()

print("\n--- Step 3: Sequential Forward Selection (SFS) ---")
# We will select the top 2 features
sfs = SequentialFeatureSelector(model, n_features_to_select=2, direction='forward')
sfs.fit(X_train, y_train)

sfs_features = X.columns[sfs.get_support()].tolist()
print(f"Top Features selected by SFS: {sfs_features}")

# Train and evaluate SFS model
X_train_sfs = sfs.transform(X_train)
X_test_sfs = sfs.transform(X_test)
model.fit(X_train_sfs, y_train)
sfs_acc = accuracy_score(y_test, model.predict(X_test_sfs))

print("\n--- Step 4: Sequential Backward Elimination (SBE) ---")
# We will select the top 2 features using backward direction
sbe = SequentialFeatureSelector(model, n_features_to_select=2, direction='backward')
sbe.fit(X_train, y_train)

sbe_features = X.columns[sbe.get_support()].tolist()
print(f"Top Features selected by SBE: {sbe_features}")

# Train and evaluate SBE model
X_train_sbe = sbe.transform(X_train)
X_test_sbe = sbe.transform(X_test)
model.fit(X_train_sbe, y_train)
sbe_acc = accuracy_score(y_test, model.predict(X_test_sbe))

In [ ]:
# 4. Compare Performance
print("\n--- Step 5: Performance Comparison ---")
comparison_df = pd.DataFrame({
    'Method': ['Forward Selection (SFS)', 'Backward Elimination (SBE)'],
    'Selected Features': [", ".join(sfs_features), ", ".join(sbe_features)],
    'Accuracy Score': [sfs_acc, sbe_acc]
})

print(comparison_df)

# Visualizing the comparison
plt.figure(figsize=(8, 5))
plt.bar(comparison_df['Method'], comparison_df['Accuracy Score'], color=['skyblue', 'salmon'])
plt.ylim(0, 1.1)
plt.ylabel('Accuracy Score')
plt.title('Comparison of SFS vs SBE Model Performance')
for i, v in enumerate(comparison_df['Accuracy Score']):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')
plt.show()